# Lacrosse field-landmark detector — GPU training

Trains the field-markings detector on synthetic lacrosse data and gives you the finished model to hand back to Claude. No token or login needed (the repo is public).

**Step 1:** `Runtime → Change runtime type → GPU` (a free T4 is fine).
**Step 2:** `Runtime → Run all`.
**Step 3:** at the end it downloads `models_cal.zip` — send that file to Claude in the chat.

Keep this tab open and connected while it runs (~1 hour). Nothing runs until you start it.


### 1. Confirm a GPU is attached


In [ ]:
import torch, subprocess
print(subprocess.run(['nvidia-smi'],capture_output=True,text=True).stdout or 'nvidia-smi not found')
assert torch.cuda.is_available(), 'No GPU. Runtime -> Change runtime type -> GPU, then Run all again.'
print('GPU ready:', torch.cuda.get_device_name(0))


### 2. Get the code (public repo, no login)


In [ ]:
%%bash
rm -rf /content/lax-vision
git clone --depth 1 https://github.com/mooneybg19/lax-vision.git /content/lax-vision
echo 'cloned'


### 3. Install dependencies (Colab already has GPU PyTorch)


In [ ]:
!pip -q install opencv-python-headless scipy numpy


### 4. Render the synthetic lacrosse-field training data
No hand-labeling — the field geometry is projected through many simulated normal-film camera angles.


In [ ]:
!cd /content/lax-vision && python pipeline/gen_synth_field.py synth --n 4000 --preview 8


### 5. Train the detector on the GPU
The 25 epochs take roughly 30–60 min on a T4. You can raise `--bs` to 32 on a GPU to go faster.


In [ ]:
!cd /content/lax-vision && python pipeline/train_field_kp.py synth models_cal --epochs 25 --bs 16


### 6. Evaluate on held-out synthetic cameras


In [ ]:
!cd /content/lax-vision && python pipeline/eval_cal.py models_cal/field_kp.pt | tee models_cal/eval.txt


### 7. Download the trained model
This saves `models_cal.zip` to your computer. Send that file to Claude in the chat and Claude wires it into the engine.


In [ ]:
from google.colab import files
!cd /content/lax-vision && cp synth/preview_000000.jpg models_cal/ 2>/dev/null; zip -qr /content/models_cal.zip models_cal
files.download('/content/models_cal.zip')


---
When Claude has `models_cal.zip`, the next step is to wire the model into the engine's field step and re-run the dev game to see whether the field fit — and the sets, events and player stats on top of it — improved.
